# Cross-cookie

На последнем фолде было около 0.635

Значит сначала сделаем обычные признаки еще полнее. Потом добавим инфу о том, насколько одни и те же значения повторяются у разных куки

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from catboost import CatBoostClassifier

RANDOM_STATE = 2026

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from metric import precision_at_recall
from src.features_base import (
    filter_events_by_window,
    prepare_events,
    build_base_features,
)
from src.features_cross import (
    FREQUENCY_COLUMNS,
    build_cross_cookie_features,
    percentile_rank,
)

print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: /Users/an.m.titova/Documents/bot_detection_case


In [2]:
train = pd.read_csv(
    DATA_DIR / "train.csv",
    parse_dates=[
        "cookie_created_at",
        "window_start_ts",
        "window_end_ts",
    ],
)

test = pd.read_csv(
    DATA_DIR / "test.csv",
    parse_dates=[
        "cookie_created_at",
        "window_start_ts",
        "window_end_ts",
    ],
)

events = pd.read_csv(
    DATA_DIR / "events.csv.gz",
    parse_dates=["event_ts"],
)

print("train:", train.shape)
print("test:", test.shape)
print("events:", events.shape)

assert train["cookie_id"].is_unique
assert test["cookie_id"].is_unique
assert train["target"].isin([0, 1]).all()


train: (11091, 5)
test: (4909, 4)
events: (328905, 14)


In [3]:
events_train = filter_events_by_window(
    events,
    train,
)

events_test = filter_events_by_window(
    events,
    test,
)

events_train = prepare_events(events_train)
events_test = prepare_events(events_test)

print("train events in windows:", len(events_train))
print("test events in windows:", len(events_test))


train events in windows: 198436
test events in windows: 89690


In [4]:
EVENT_TYPES = sorted(
    events_train["event_name"]
    .dropna()
    .unique()
)

PLATFORM_TYPES = sorted(
    events_train["platform_clean"]
    .dropna()
    .unique()
)

print("EVENT_TYPES:", EVENT_TYPES)
print("PLATFORM_TYPES:", PLATFORM_TYPES)
print("FREQUENCY_COLUMNS:", FREQUENCY_COLUMNS)


EVENT_TYPES: ['contact_chat_open', 'contact_message_sent', 'contact_phone_show', 'favorite_add', 'item_view', 'login', 'photo_swipe', 'search_results_view', 'seller_page_view']
PLATFORM_TYPES: ['android', 'desktop', 'ios', 'iphone', 'web']
FREQUENCY_COLUMNS: ['user_agent', 'item_id', 'search_query', 'item_category', 'item_location']


In [5]:
train_base = build_base_features(
    train,
    events_train,
    EVENT_TYPES,
    PLATFORM_TYPES,
)

test_base = build_base_features(
    test,
    events_test,
    EVENT_TYPES,
    PLATFORM_TYPES,
)

train_base = train_base.merge(
    train[["cookie_id", "target"]],
    on="cookie_id",
    how="left",
)

assert len(train_base) == len(train)
assert len(test_base) == len(test)
assert train_base["cookie_id"].is_unique
assert test_base["cookie_id"].is_unique

train_schema = [
    col
    for col in train_base.columns
    if col != "target"
]

assert train_schema == test_base.columns.tolist()

NON_FEATURE_COLS = [
    "cookie_id",
    "target",
    "cookie_created_at",
    "window_start_ts",
    "window_end_ts",
]

base_feature_cols = [
    col
    for col in train_base.columns
    if col not in NON_FEATURE_COLS
]

print("Train base:", train_base.shape)
print("Test base:", test_base.shape)
print("Base model features:", len(base_feature_cols))


/Users/an.m.titova/Documents/bot_detection_case/src/features_base.py:489: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features[f"share_platform_{platform}"] = (
/Users/an.m.titova/Documents/bot_detection_case/src/features_base.py:879: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features["events_per_active_hour"] = (
/Users/an.m.titova/Documents/bot_detection_case/src/features_base.py:890: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perfo

Train base: (11091, 195)
Test base: (4909, 194)
Base model features: 190


/Users/an.m.titova/Documents/bot_detection_case/src/features_base.py:879: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features["events_per_active_hour"] = (
/Users/an.m.titova/Documents/bot_detection_case/src/features_base.py:890: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features["unique_items_per_item_view"] = safe_ratio(
/Users/an.m.titova/Documents/bot_detection_case/src/features_base.py:899: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has 

В `features_base.py` собрала более полный набор обычных признаков.

In [6]:
FOLDS = [
    ("2026-04-13", "2026-04-15"),
    ("2026-04-15", "2026-04-17"),
    ("2026-04-17", "2026-04-20"),
]

for valid_start, valid_end in FOLDS:
    train_mask = (
        train_base["window_start_ts"]
        < pd.Timestamp(valid_start)
    )

    valid_mask = (
        (train_base["window_start_ts"] >= pd.Timestamp(valid_start))
        & (train_base["window_start_ts"] < pd.Timestamp(valid_end))
    )

    print(
        valid_start,
        valid_end,
        "train =", int(train_mask.sum()),
        "valid =", int(valid_mask.sum()),
        "bots =", int(train_base.loc[valid_mask, "target"].sum()),
    )


2026-04-13 2026-04-15 train = 5930 valid = 1669 bots = 123
2026-04-15 2026-04-17 train = 7599 valid = 1541 bots = 125
2026-04-17 2026-04-20 train = 9140 valid = 1951 bots = 160


In [7]:
def make_model(model_name):
    if model_name == "HistGradientBoosting":
        return HistGradientBoostingClassifier(
            learning_rate=0.045,
            max_iter=350,
            max_leaf_nodes=31,
            min_samples_leaf=20,
            l2_regularization=1.0,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )

    if model_name == "CatBoost":
        return CatBoostClassifier(
            iterations=460,
            depth=7,
            learning_rate=0.03,
            l2_leaf_reg=8.0,
            class_weights=[1.0, 4.0],
            loss_function="Logloss",
            random_seed=RANDOM_STATE,
            random_strength=1.0,
            bagging_temperature=1.0,
            border_count=128,
            allow_writing_files=False,
            verbose=False,
        )

    raise ValueError(model_name)


In [8]:
base_results = []

for valid_start, valid_end in FOLDS:
    train_mask = (
        train_base["window_start_ts"]
        < pd.Timestamp(valid_start)
    )

    valid_mask = (
        (train_base["window_start_ts"] >= pd.Timestamp(valid_start))
        & (train_base["window_start_ts"] < pd.Timestamp(valid_end))
    )

    X_train = train_base.loc[
        train_mask,
        base_feature_cols,
    ]

    y_train = train_base.loc[
        train_mask,
        "target",
    ]

    X_valid = train_base.loc[
        valid_mask,
        base_feature_cols,
    ]

    y_valid = train_base.loc[
        valid_mask,
        "target",
    ]

    for model_name in [
        "HistGradientBoosting",
        "CatBoost",
    ]:
        model = make_model(model_name)
        model.fit(X_train, y_train)

        score = model.predict_proba(X_valid)[:, 1]

        base_results.append(
            {
                "feature_set": "base",
                "model": model_name,
                "valid_start": valid_start,
                "valid_end": valid_end,
                "train_size": len(X_train),
                "valid_size": len(X_valid),
                "n_bots_valid": int(y_valid.sum()),
                "p_at_r_70": precision_at_recall(
                    y_valid,
                    score,
                ),
                "roc_auc": roc_auc_score(
                    y_valid,
                    score,
                ),
                "pr_auc": average_precision_score(
                    y_valid,
                    score,
                ),
            }
        )

base_results = pd.DataFrame(base_results)

base_results


,feature_set,model,valid_start,valid_end,train_size,valid_size,n_bots_valid,p_at_r_70,roc_auc,pr_auc
0,base,HistGradientBoosting,2026-04-13,2026-04-15,5930,1669,123,0.756522,0.936874,0.780507
1,base,CatBoost,2026-04-13,2026-04-15,5930,1669,123,0.737288,0.930968,0.762279
2,base,HistGradientBoosting,2026-04-15,2026-04-17,7599,1541,125,0.765217,0.938644,0.782381
3,base,CatBoost,2026-04-15,2026-04-17,7599,1541,125,0.769231,0.947559,0.805422
4,base,HistGradientBoosting,2026-04-17,2026-04-20,9140,1951,160,0.761905,0.935099,0.787156
5,base,CatBoost,2026-04-17,2026-04-20,9140,1951,160,0.783217,0.935431,0.779513


In [9]:
base_summary = (
    base_results
    .groupby("model")[
        [
            "p_at_r_70",
            "roc_auc",
            "pr_auc",
        ]
    ]
    .agg(["mean", "std"])
)

base_summary


p_at_r_70             roc_auc              pr_auc  \
                          mean       std      mean       std      mean   
model                                                                    
CatBoost              0.763245  0.023542  0.937986  0.008586  0.782405   
HistGradientBoosting  0.761215  0.004389  0.936872  0.001772  0.783348   

                                
                           std  
model                           
CatBoost              0.021716  
HistGradientBoosting  0.003428

Даже без cross-cookie средний P@R>=70% около 0.76, а на последнем фолде обе модели примерно 0.783
То есть большой прирост пришел просто от более полного набора нормальных признаков

Для `user_agent`, `item_id`, `search_query`, `item_category` и `item_location` смотрю, у скольких других cookie встречалось такое же значение
Тут очень легко сделать утечку данных, поэтому для validation беру частоты только из более раннего train. Для train еще вычитаю саму куки из частоты

In [10]:
cross_results = []

for valid_start, valid_end in FOLDS:
    train_mask = (
        train_base["window_start_ts"]
        < pd.Timestamp(valid_start)
    )

    valid_mask = (
        (train_base["window_start_ts"] >= pd.Timestamp(valid_start))
        & (train_base["window_start_ts"] < pd.Timestamp(valid_end))
    )

    train_ids = train_base.loc[
        train_mask,
        "cookie_id",
    ]

    valid_ids = train_base.loc[
        valid_mask,
        "cookie_id",
    ]

    reference_events = events_train.loc[
        events_train["cookie_id"].isin(train_ids)
    ]

    valid_events = events_train.loc[
        events_train["cookie_id"].isin(valid_ids)
    ]

    train_cross = build_cross_cookie_features(
        reference_events=reference_events,
        target_events=reference_events,
        target_cookie_ids=train_ids,
        leave_one_cookie_out=True,
    )

    valid_cross = build_cross_cookie_features(
        reference_events=reference_events,
        target_events=valid_events,
        target_cookie_ids=valid_ids,
        leave_one_cookie_out=False,
    )

    train_fold = (
        train_base.loc[train_mask]
        .merge(
            train_cross,
            on="cookie_id",
            how="left",
            validate="one_to_one",
        )
    )

    valid_fold = (
        train_base.loc[valid_mask]
        .merge(
            valid_cross,
            on="cookie_id",
            how="left",
            validate="one_to_one",
        )
    )

    model_features = [
        col
        for col in train_fold.columns
        if col not in NON_FEATURE_COLS
    ]

    assert model_features == [
        col
        for col in valid_fold.columns
        if col not in NON_FEATURE_COLS
    ]

    X_train = train_fold[model_features]
    y_train = train_fold["target"]

    X_valid = valid_fold[model_features]
    y_valid = valid_fold["target"]

    fold_scores = {}

    for model_name in [
        "HistGradientBoosting",
        "CatBoost",
    ]:
        model = make_model(model_name)
        model.fit(X_train, y_train)

        score = model.predict_proba(X_valid)[:, 1]
        fold_scores[model_name] = score

        cross_results.append(
            {
                "feature_set": "base_plus_cross",
                "model": model_name,
                "valid_start": valid_start,
                "valid_end": valid_end,
                "train_size": len(X_train),
                "valid_size": len(X_valid),
                "n_bots_valid": int(y_valid.sum()),
                "n_features": len(model_features),
                "p_at_r_70": precision_at_recall(
                    y_valid,
                    score,
                ),
                "roc_auc": roc_auc_score(
                    y_valid,
                    score,
                ),
                "pr_auc": average_precision_score(
                    y_valid,
                    score,
                ),
            }
        )

    blend_score = (
        0.55
        * percentile_rank(
            fold_scores["HistGradientBoosting"]
        )
        + 0.45
        * percentile_rank(
            fold_scores["CatBoost"]
        )
    )

    cross_results.append(
        {
            "feature_set": "base_plus_cross",
            "model": "RankBlend_55HGB_45CAT",
            "valid_start": valid_start,
            "valid_end": valid_end,
            "train_size": len(X_train),
            "valid_size": len(X_valid),
            "n_bots_valid": int(y_valid.sum()),
            "n_features": len(model_features),
            "p_at_r_70": precision_at_recall(
                y_valid,
                blend_score,
            ),
            "roc_auc": roc_auc_score(
                y_valid,
                blend_score,
            ),
            "pr_auc": average_precision_score(
                y_valid,
                blend_score,
            ),
        }
    )

cross_results = pd.DataFrame(cross_results)

cross_results.sort_values(
    ["valid_start", "model"]
)


,feature_set,model,valid_start,valid_end,train_size,valid_size,n_bots_valid,n_features,p_at_r_70,roc_auc,pr_auc
1,base_plus_cross,CatBoost,2026-04-13,2026-04-15,5930,1669,123,250,0.870000,0.942427,0.803450
0,base_plus_cross,HistGradientBoosting,2026-04-13,2026-04-15,5930,1669,123,250,0.862745,0.935596,0.800318
2,base_plus_cross,RankBlend_55HGB_45CAT,2026-04-13,2026-04-15,5930,1669,123,250,0.890000,0.941493,0.806988
4,base_plus_cross,CatBoost,2026-04-15,2026-04-17,7599,1541,125,250,0.888889,0.948723,0.840418
3,base_plus_cross,HistGradientBoosting,2026-04-15,2026-04-17,7599,1541,125,250,0.871287,0.949367,0.827078
5,base_plus_cross,RankBlend_55HGB_45CAT,2026-04-15,2026-04-17,7599,1541,125,250,0.897959,0.950712,0.841502
7,base_plus_cross,CatBoost,2026-04-17,2026-04-20,9140,1951,160,250,0.829630,0.938990,0.784607
6,base_plus_cross,HistGradientBoosting,2026-04-17,2026-04-20,9140,1951,160,250,0.870229,0.943579,0.801407
8,base_plus_cross,RankBlend_55HGB_45CAT,2026-04-17,2026-04-20,9140,1951,160,250,0.869231,0.943609,0.800186


In [11]:
cross_summary = (
    cross_results
    .groupby("model")[
        [
            "p_at_r_70",
            "roc_auc",
            "pr_auc",
        ]
    ]
    .agg(["mean", "std", "min", "max"])
)

cross_summary


p_at_r_70                                 roc_auc  \
                           mean       std       min       max      mean   
model                                                                     
CatBoost               0.862840  0.030272  0.829630  0.888889  0.943380   
HistGradientBoosting   0.868087  0.004656  0.862745  0.871287  0.942847   
RankBlend_55HGB_45CAT  0.885730  0.014833  0.869231  0.897959  0.945271   

                                                       pr_auc            \
                            std       min       max      mean       std   
model                                                                     
CatBoost               0.004936  0.938990  0.948723  0.809492  0.028392   
HistGradientBoosting   0.006915  0.935596  0.949367  0.809601  0.015145   
RankBlend_55HGB_45CAT  0.004829  0.941493  0.950712  0.816225  0.022153   

                                           
                            min       max  
model                                      
CatBoost               0.784607  0.840418  
HistGradientBoosting   0.800318  0.827078  
RankBlend_55HGB_45CAT  0.800186  0.841502

Blend лучший в среднем и на всех трех фолдах остается сильно выше 0.7

In [12]:
comparison = pd.concat(
    [
        base_results[
            [
                "feature_set",
                "model",
                "valid_start",
                "valid_end",
                "p_at_r_70",
                "roc_auc",
                "pr_auc",
            ]
        ],
        cross_results[
            [
                "feature_set",
                "model",
                "valid_start",
                "valid_end",
                "p_at_r_70",
                "roc_auc",
                "pr_auc",
            ]
        ],
    ],
    ignore_index=True,
)

comparison.sort_values(
    ["valid_start", "feature_set", "model"]
)


,feature_set,model,valid_start,valid_end,p_at_r_70,roc_auc,pr_auc
1,base,CatBoost,2026-04-13,2026-04-15,0.737288,0.930968,0.762279
0,base,HistGradientBoosting,2026-04-13,2026-04-15,0.756522,0.936874,0.780507
7,base_plus_cross,CatBoost,2026-04-13,2026-04-15,0.870000,0.942427,0.803450
6,base_plus_cross,HistGradientBoosting,2026-04-13,2026-04-15,0.862745,0.935596,0.800318
8,base_plus_cross,RankBlend_55HGB_45CAT,2026-04-13,2026-04-15,0.890000,0.941493,0.806988
3,base,CatBoost,2026-04-15,2026-04-17,0.769231,0.947559,0.805422
2,base,HistGradientBoosting,2026-04-15,2026-04-17,0.765217,0.938644,0.782381
10,base_plus_cross,CatBoost,2026-04-15,2026-04-17,0.888889,0.948723,0.840418
9,base_plus_cross,HistGradientBoosting,2026-04-15,2026-04-17,0.871287,0.949367,0.827078
11,base_plus_cross,RankBlend_55HGB_45CAT,2026-04-15,2026-04-17,0.897959,0.950712,0.841502


Смешиваю модели по rank, а не по самим вероятностям. У CatBoost и HGB вероятности могут быть в разных масштабах, а нашей метрике в основном важен порядок куки
В среднем blend оказался лучше каждой модели отдельно, поэтому дальше беру его

In [13]:
base_results.to_csv(
    OUTPUT_DIR / "base_temporal_cv_results.csv",
    index=False,
)

cross_results.to_csv(
    OUTPUT_DIR / "cross_cookie_temporal_cv_results.csv",
    index=False,
)

comparison.to_csv(
    OUTPUT_DIR / "model_comparison_results.csv",
    index=False,
)

print("Saved aggregate metrics to:", OUTPUT_DIR)


Saved aggregate metrics to: /Users/an.m.titova/Documents/bot_detection_case/outputs


Тут можно закончить, дальше уже не будем добавлять признаки просто ради количества. Беру то, что проверено на трех временных фолдах, и обучаю финальную модель на всем train